# 08 — Adaptive Forecast Ensemble vs Random Walk

This notebook combines the frozen **Random Walk**, **Statistical**, and **Machine-Learning** forecasts using only forecast errors that were already observable at each forecast origin.

At date `t`, a past forecast may influence the weights only when its target date is already known:

`Target_Date <= Forecast_Origin(t)`

Adaptive strategies tested:
- Random Walk only
- recent champion model
- inverse-RMSE weights
- convex non-negative RMSE-minimizing weights

For each outer year, the strategy is chosen using **earlier outer years only**. 2019 is a warm-up year and uses Random Walk. The 2024–2025 final holdout is never loaded.

Core metrics: **RMSE, MAE, Forecast Skill vs RW, Theil's U, DM p-value**.

In [ ]:
from pathlib import Path
import warnings, time
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",180)
pd.set_option("display.width",220)
pd.set_option("display.float_format",lambda x:f"{x:,.6f}")

CURRENCIES=["USD","GBP","EUR","JPY"]
HORIZONS=[1,3,7]
YEARS=[2019,2020,2021,2022,2023]
MIN_HISTORY=20
WEIGHT_STEP=0.10

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        rw=root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        st=root/"outputs"/"predictions"/"05_statistical_predictions_2019_2023.csv"
        if rw.exists() and st.exists(): return root.resolve()
    raise FileNotFoundError("Script 04 and 05 prediction files not found.")

ROOT=find_root(); PRED_DIR=ROOT/"outputs"/"predictions"; TABLE_DIR=ROOT/"outputs"/"tables"
TABLE_DIR.mkdir(parents=True,exist_ok=True)
RW_PATH=PRED_DIR/"04_random_walk_predictions_2019_2023.csv"
STAT_PATH=PRED_DIR/"05_statistical_predictions_2019_2023.csv"
ml_candidates=[PRED_DIR/"06_FAST_ml_predictions_2019_2023.csv",PRED_DIR/"06_ml_predictions_2019_2023.csv"]
ML_PATH=next((p for p in ml_candidates if p.exists()),None)
if ML_PATH is None: raise FileNotFoundError("No Script 06 ML prediction file found.")
print("RW:",RW_PATH.name); print("STAT:",STAT_PATH.name); print("ML:",ML_PATH.name)

In [ ]:
rw=pd.read_csv(RW_PATH,parse_dates=["Forecast_Origin","Target_Date"])
stat=pd.read_csv(STAT_PATH,parse_dates=["Forecast_Origin","Target_Date"])
ml=pd.read_csv(ML_PATH,parse_dates=["Forecast_Origin","Target_Date"])

if "Stat_Prediction" not in stat.columns: raise ValueError("Stat_Prediction missing")
if "ML_Prediction" not in ml.columns: raise ValueError("ML_Prediction missing")

keys=["Currency","Horizon","Forecast_Origin"]
base=rw[keys+["Target_Date","Actual","RW_Prediction"]].copy()
base=base.merge(stat[keys+["Stat_Prediction"]],on=keys,how="inner")
base=base.merge(ml[keys+["ML_Prediction"]],on=keys,how="inner")
base=base.sort_values(keys).reset_index(drop=True)
base["Validation_Year"]=base["Forecast_Origin"].dt.year
assert base["Forecast_Origin"].max()<pd.Timestamp("2024-01-01")
print("Matched rows:",len(base)); display(base.head())

In [ ]:
def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan

def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan

def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan

def dm_test(y,model,rw_pred,horizon=1):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    mask=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred)
    y=y[mask]; model=model[mask]; rw_pred=rw_pred[mask]; n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2; lag=max(0,int(horizon)-1); lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        cov=np.cov(d[k:],d[:-k],ddof=1)[0,1]; lrv += 2*(1-k/(lag+1))*cov
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n); p=2*student_t.sf(abs(stat),df=n-1)
    return float(stat),float(p)

In [ ]:
vals=np.round(np.arange(0,1+WEIGHT_STEP/2,WEIGHT_STEP),10)
WEIGHT_GRID=[]
for wr in vals:
    for ws in vals:
        wm=1-wr-ws
        if wm>=-1e-10 and wm<=1+1e-10:
            WEIGHT_GRID.append((float(wr),float(ws),max(0.0,float(np.round(wm,10)))))
print("Weight combinations:",len(WEIGHT_GRID))

In [ ]:
MODEL_COLS={"RW":"RW_Prediction","STAT":"Stat_Prediction","ML":"ML_Prediction"}

def rw_weights(): return {"w_RW":1.0,"w_STAT":0.0,"w_ML":0.0}

def recent_champion(hist,window):
    if len(hist)<MIN_HISTORY: return rw_weights()
    h=hist.tail(window); scores={k:rmse(h["Actual"],h[c]) for k,c in MODEL_COLS.items()}
    best=min(scores,key=scores.get); out={"w_RW":0.0,"w_STAT":0.0,"w_ML":0.0}; out[f"w_{best}"]=1.0; return out

def inverse_weights(hist,window):
    if len(hist)<MIN_HISTORY: return rw_weights()
    h=hist.tail(window); inv={}
    for k,c in MODEL_COLS.items():
        s=rmse(h["Actual"],h[c]); inv[k]=0.0 if (not np.isfinite(s) or s<=0) else 1/s
    total=sum(inv.values())
    if total<=0: return rw_weights()
    return {"w_RW":inv["RW"]/total,"w_STAT":inv["STAT"]/total,"w_ML":inv["ML"]/total}

def convex_weights(hist,window):
    if len(hist)<MIN_HISTORY: return rw_weights()
    h=hist.tail(window); y=h["Actual"].to_numpy(float); rp=h["RW_Prediction"].to_numpy(float); sp=h["Stat_Prediction"].to_numpy(float); mp=h["ML_Prediction"].to_numpy(float)
    best=(1.0,0.0,0.0); best_score=np.inf
    for wr,ws,wm in WEIGHT_GRID:
        p=wr*rp+ws*sp+wm*mp; s=rmse(y,p)
        if s<best_score: best_score=s; best=(wr,ws,wm)
    return {"w_RW":best[0],"w_STAT":best[1],"w_ML":best[2]}

STRATEGIES=["RW_ONLY","CHAMPION_40","CHAMPION_80","INV_40","INV_80","INV_160","CONVEX_40","CONVEX_80","CONVEX_160"]

def get_weights(strategy,hist):
    if strategy=="RW_ONLY": return rw_weights()
    if strategy.startswith("CHAMPION_"): return recent_champion(hist,int(strategy.split("_")[-1]))
    if strategy.startswith("INV_"): return inverse_weights(hist,int(strategy.split("_")[-1]))
    if strategy.startswith("CONVEX_"): return convex_weights(hist,int(strategy.split("_")[-1]))
    raise ValueError(strategy)

In [ ]:
def simulate_strategy(task,strategy):
    task=task.sort_values("Forecast_Origin").reset_index(drop=True); rows=[]
    for _,r in task.iterrows():
        origin=r["Forecast_Origin"]
        hist=task[(task["Target_Date"]<=origin)&(task["Forecast_Origin"]<origin)].copy()
        w=get_weights(strategy,hist)
        pred=w["w_RW"]*r["RW_Prediction"]+w["w_STAT"]*r["Stat_Prediction"]+w["w_ML"]*r["ML_Prediction"]
        rows.append({"Currency":r["Currency"],"Horizon":r["Horizon"],"Forecast_Origin":origin,"Target_Date":r["Target_Date"],"Validation_Year":r["Validation_Year"],"Actual":r["Actual"],"RW_Prediction":r["RW_Prediction"],"Stat_Prediction":r["Stat_Prediction"],"ML_Prediction":r["ML_Prediction"],"Strategy":strategy,"Ensemble_Prediction":pred,**w,"Resolved_History_Rows":len(hist)})
    return pd.DataFrame(rows)

all_s=[]; t0=time.time()
for cur in CURRENCIES:
    for h in HORIZONS:
        task=base[(base["Currency"]==cur)&(base["Horizon"]==h)].copy()
        print(f"Simulating {cur} H{h}...",flush=True)
        for s in STRATEGIES: all_s.append(simulate_strategy(task,s))
strategy_predictions=pd.concat(all_s,ignore_index=True)
print(f"Finished in {time.time()-t0:.1f}s")

In [ ]:
selection_rows=[]; final_frames=[]
for cur in CURRENCIES:
    for h in HORIZONS:
        task_all=strategy_predictions[(strategy_predictions["Currency"]==cur)&(strategy_predictions["Horizon"]==h)].copy()
        for year in YEARS:
            yr=task_all[task_all["Validation_Year"]==year].copy()
            if year==2019:
                selected="RW_ONLY"; hist_rmse=np.nan
            else:
                prior=task_all[task_all["Validation_Year"]<year].copy()
                scores=[]
                for s,g in prior.groupby("Strategy"):
                    scores.append((s,rmse(g["Actual"],g["Ensemble_Prediction"])))
                scores=sorted(scores,key=lambda x:(x[1],x[0])); selected,hist_rmse=scores[0]
            chosen=yr[yr["Strategy"]==selected].copy(); chosen["Selected_Strategy"]=selected; final_frames.append(chosen)
            selection_rows.append({"Currency":cur,"Horizon":h,"Outer_Year":year,"Selected_Strategy":selected,"Historical_Selection_RMSE":hist_rmse})
adaptive_predictions=pd.concat(final_frames,ignore_index=True)
strategy_selections=pd.DataFrame(selection_rows)
display(strategy_selections)

In [ ]:
yearly=[]
for (cur,h,year),g in adaptive_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    er=rmse(g["Actual"],g["Ensemble_Prediction"]); ea=mae(g["Actual"],g["Ensemble_Prediction"]); rr=rmse(g["Actual"],g["RW_Prediction"]); _,p=dm_test(g["Actual"],g["Ensemble_Prediction"],g["RW_Prediction"],h)
    yearly.append({"Currency":cur,"Horizon":int(h),"Validation_Year":int(year),"Observations":len(g),"RMSE":er,"MAE":ea,"Forecast_Skill_vs_RW":skill(er,rr),"Theils_U":theils_u(er,rr),"DM_pvalue_vs_RW":p})
yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"])
display(yearly_results); yearly_results.to_csv(TABLE_DIR/"08_adaptive_ensemble_yearly_results.csv",index=False)

In [ ]:
pooled=[]
for (cur,h),g in adaptive_predictions.groupby(["Currency","Horizon"]):
    er=rmse(g["Actual"],g["Ensemble_Prediction"]); ea=mae(g["Actual"],g["Ensemble_Prediction"]); rr=rmse(g["Actual"],g["RW_Prediction"]); _,p=dm_test(g["Actual"],g["Ensemble_Prediction"],g["RW_Prediction"],h)
    pooled.append({"Currency":cur,"Horizon":int(h),"Observations":len(g),"RMSE":er,"MAE":ea,"Forecast_Skill_vs_RW":skill(er,rr),"Theils_U":theils_u(er,rr),"DM_pvalue_vs_RW":p})
pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"])
display(pooled_results); pooled_results.to_csv(TABLE_DIR/"08_adaptive_ensemble_pooled_results.csv",index=False)
wins=pooled_results.assign(Beats_RW=pooled_results["Forecast_Skill_vs_RW"]>0,DM_Significant_5pct=pooled_results["DM_pvalue_vs_RW"]<0.05)
display(wins); print("Tasks beating RW:",int(wins["Beats_RW"].sum()),"/",len(wins))

In [ ]:
weight_summary=(adaptive_predictions.groupby(["Currency","Horizon"]).agg(Mean_RW_Weight=("w_RW","mean"),Mean_STAT_Weight=("w_STAT","mean"),Mean_ML_Weight=("w_ML","mean")).reset_index())
display(weight_summary)
adaptive_predictions.to_csv(PRED_DIR/"08_adaptive_ensemble_predictions_2019_2023.csv",index=False)
strategy_selections.to_csv(TABLE_DIR/"08_strategy_selections.csv",index=False)
weight_summary.to_csv(TABLE_DIR/"08_adaptive_weight_summary.csv",index=False)

In [ ]:
base_keys=set(zip(base["Currency"],base["Horizon"],base["Forecast_Origin"]))
final_keys=set(zip(adaptive_predictions["Currency"],adaptive_predictions["Horizon"],adaptive_predictions["Forecast_Origin"]))
quality_gate=pd.DataFrame([
    {"Area":"Final holdout isolation","Status":"PASS","Evidence":"Only 2019–2023 predictions loaded"},
    {"Area":"Resolved-error rule","Status":"PASS","Evidence":"Weights use only forecasts whose target dates are already observed"},
    {"Area":"Strategy selection","Status":"PASS","Evidence":"Each outer year selected from earlier outer years only"},
    {"Area":"Convex weights","Status":"PASS","Evidence":"Non-negative weights sum to one"},
    {"Area":"Matched origins","Status":"PASS" if base_keys==final_keys else "REVIEW","Evidence":f"Base={len(base_keys)}, Adaptive={len(final_keys)}"}
])
display(quality_gate); quality_gate.to_csv(TABLE_DIR/"08_quality_gate.csv",index=False)

In [ ]:
print("="*82)
print("SCRIPT 08 — ADAPTIVE FORECAST ENSEMBLE COMPLETE")
print("="*82)
display(pooled_results)
print("\nTasks beating RW:",int((pooled_results["Forecast_Skill_vs_RW"]>0).sum()),"/",len(pooled_results))
print("Forecast Skill > 0 and Theil U < 1 = lower RMSE than RW.")
print("DM p < 0.05 strengthens a positive result only when skill is positive.")
print("2024–2025 remains untouched.")